# 02 — 2.5D baseline (IDEAS #14)

The first real imaging model, and the reference point every later experiment is measured against.

**Architecture.** A `timm` backbone runs per slice; slices are pooled per series with gated
attention; the 3 slot embeddings are concatenated and a single linear head emits 12 logits.
Each slice enters as 3 channels of `(i-1, i, i+1)` — the cheap 2.5D trick that gives the
2D backbone a little through-plane context without a 3D conv in sight.

**Targets.** LLM-extracted weak probabilities (`train_weak_labels.csv`), used as *soft* BCE
targets rather than thresholded — a 0.55 and a 0.95 are not the same claim. The 58 gold rows
carry 2x sample weight.

**Validation.** Per-label ROC AUC on the held-out fold, against binarized weak labels
(`> 0.5`) and separately against the gold rows in that fold. Score is the unweighted mean of
the 12 AUCs, matching the competition metric (COMPETITION_SPEC §3).

**Inputs.** The cache from `03_cache_builder_v3.ipynb` (`rsna-knee-cache-v3`) plus the repo CSVs (`folds.csv`,
`weak_labels/train_weak_labels.csv`) uploaded as a small dataset.

## Config

In [ ]:
# Every path is here. Adjust the two input mounts after the datasets are created.
CFG = {
    # Cache dataset published by 03_cache_builder_v3.ipynb (one <study>.npz per study).
    # Cache: every *.npz under any /kaggle/input mount whose path contains CACHE_KEY
    # (v2 = one kernel "rsna-knee-cache-builder"; v3 = shards "rsna-knee-cache-v3-s0/1/2").
    # Slot count, slices and image size are read from the files, not configured.
    "CACHE_KEY": "rsna-knee-cache-v3",
    # Small dataset holding the repo CSVs: folds.csv and train_weak_labels.csv.
    "CSV_DIR": "/kaggle/input/knee-csvs",
    "FOLDS_CSV": "folds.csv",
    "WEAK_CSV": "train_weak_labels.csv",
    "OUT_DIR": "/kaggle/working",

    "FOLD": 0,
    "N_FOLDS": 5,

    "BACKBONE": "convnext_tiny",   # timm name; pretrained. ViT option: "vit_small_patch14_dinov2.lvd142m"
    "PRETRAINED": True,
    # ViT-only settings (ignored for CNNs): input side (multiple of the 14 px patch), how many
    # of the last blocks train (rest frozen), layer-wise LR decay, and separate LRs.
    "VIT_IMG": 224,
    "MODEL_IMG": None,
    "VIT_UNFREEZE": 6,
    "VIT_LLRD": 0.78,
    "VIT_LR_BACKBONE": 1.5e-5,
    "VIT_LR_HEAD": 1e-3,
    "IMG": None,                   # filled from the cache
    "N_SLOTS": None,               # filled from the cache
    "CACHE_SLICES": None,          # filled from the cache
    "SLOTS": None,                 # cache slot indices to use; None = all.
                                   # v3 order: 0 sagFS 1 corFS 2 axFS 3 sagNonFS 4 corT1 5 sagT1.
                                   # The three non-FS slots are present in only 40-63% of
                                   # studies and zero-filled otherwise, so they can be dropped.
    "SLICE_DIR_FIX": None,         # path to a per-(study, slot) slice-direction table, or None.
                                   # Some slots are stored back-to-front relative to their
                                   # same-plane reference (sag T1 30%, sag non-FS 18%, cor T1 8%),
                                   # which makes slot k's slice i mean different anatomy study to
                                   # study. See EXP-016 and src/slice_dir_table.py.
    "SLICE_STRIDE": 2,             # every 2nd slice -> 16 per slot (v2) / 12 per slot (v3)
    "ATTN_DIM": 128,
    "HEAD": "concat",              # "concat": shared AttnPool per slot -> concat -> Linear (EXP-019).
                                   # "label_attn": one attention softmax PER LABEL over every slice
                                   # embedding in the study (attention-MIL, as the public Raptor
                                   # CoAtNet arm), so a focal finding on two slices of one slot is not
                                   # averaged away by a label-agnostic gate. See EXP-020.
    "HEAD_HIDDEN": 256,            # label_attn only: width of the attention MLP
    "SLOT_EMB": False,             # label_attn only: add a learned per-slot vector to each slice
    "POS_EMB": False,              # learned per-(slot, slice-position) vector before pooling; pair with SLICE_DIR_FIX
                                   # embedding so a label can tell which slot a slice came from
    "DROPOUT": 0.2,

    "EPOCHS": 6,
    "SWA_EPOCHS": 0,               # average the weights (and BN stats) of the last N epochs into
                                   # one extra model, scored and saved beside the best epoch. 0 = off
    "BATCH_SIZE": 2,               # studies per step; each is N_SLOTS*16 images
    "GRAD_ACCUM": 8,               # effective batch 16
    "GRAD_CHECKPOINT": True,       # ~50 images/step through the backbone; without this it OOMs
    "LR": 2e-4,
    "HEAD_LR_MULT": 10.0,          # fresh head learns faster than the pretrained trunk
    "WEIGHT_DECAY": 0.01,
    "WARMUP_FRAC": 0.05,
    # Two-stage fine-tune: for the first STAGE1_EPOCHS the backbone LR is 0 (only head +
    # attention pooling learn), then the full schedule starts for every group. 0 = off.
    "STAGE1_EPOCHS": 0,
    "GRAD_CLIP": 1.0,
    "AMP": True,
    "NUM_WORKERS": 2,
    "GOLD_WEIGHT": 3.0,            # sample weight of the 58 expert-labelled studies
    "CONF_WEIGHT": True,           # per-label weight = CONF_FLOOR + (1-CONF_FLOOR)*2|p-0.5| on weak rows
    "CONF_FLOOR": 0.25,
    # Spatial/intensity augmentation (train only), one random draw per study so every slot
    # and slice moves together. NO flips: cache v3 normalises laterality, a flip undoes it.
    "AUG": True,
    "AUG_ROT_DEG": 7.0,
    "AUG_SCALE": 0.07,
    "AUG_SHIFT": 0.05,
    "AUG_GAMMA": 0.10,

    "TIME_BUDGET_H": 5.2,          # stop cleanly inside a ~6 h session
    "SEED": 42,

    "DEBUG": False,                 # True -> 200 studies, 1 epoch. SET FALSE FOR THE REAL RUN.
    "DEBUG_STUDIES": 200,
}

import json
import os

overrides = json.loads(os.environ.get("TRAIN_OVERRIDES", "{}"))
for key, value in overrides.items():
    if key not in CFG:
        raise ValueError(f"unknown CFG key in TRAIN_OVERRIDES: {key!r}")
    CFG[key] = value
if overrides:
    print("CFG overrides:", overrides)

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

## Setup

In [ ]:
import os
import gc
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
import timm

T0 = time.time()


def seed_all(seed):
    """Seeds python, numpy and torch so a run is reproducible up to cuDNN autotuning.

    Args:
        seed: The seed given to every generator.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_all(CFG["SEED"])
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# torch >= 2.4 moved these out of torch.cuda.amp; both spellings exist on Kaggle images
# depending on the pinned version, so bind once here and use amp_* everywhere below.
try:
    from torch.amp import GradScaler as _GradScaler, autocast as _autocast

    def amp_scaler(enabled):
        """Builds a CUDA gradient scaler.

        Args:
            enabled: False makes every scaler call a no-op.

        Returns:
            A GradScaler for the CUDA device.
        """
        return _GradScaler("cuda", enabled=enabled)

    def amp_autocast(enabled):
        """Builds the mixed-precision context manager.

        Args:
            enabled: False makes the context a no-op.

        Returns:
            An autocast context manager for the CUDA device.
        """
        return _autocast("cuda", enabled=enabled)
except ImportError:
    def amp_scaler(enabled):
        """Builds a CUDA gradient scaler (torch < 2.4 spelling).

        Args:
            enabled: False makes every scaler call a no-op.

        Returns:
            A GradScaler for the CUDA device.
        """
        return torch.cuda.amp.GradScaler(enabled=enabled)

    def amp_autocast(enabled):
        """Builds the mixed-precision context manager (torch < 2.4 spelling).

        Args:
            enabled: False makes the context a no-op.

        Returns:
            An autocast context manager for the CUDA device.
        """
        return torch.cuda.amp.autocast(enabled=enabled)

USE_AMP = CFG["AMP"] and DEVICE.type == "cuda"
print("torch", torch.__version__, "| timm", timm.__version__, "| device", DEVICE)
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

if CFG["DEBUG"]:
    CFG["EPOCHS"] = 1
    print("DEBUG mode: 1 epoch,", CFG["DEBUG_STUDIES"], "studies")

## Data: cache index, folds, weak labels

In [ ]:
def find_dir(configured, must_contain):
    """Finds the directory holding a given file, starting from the configured path.

    Args:
        configured: The directory to try first.
        must_contain: A file name, or a glob pattern when it contains "*".

    Returns:
        The configured directory if it holds must_contain, else the first directory
        under /kaggle/input or data/ that does.

    Raises:
        FileNotFoundError: No directory under the search roots holds must_contain.
    """
    p = Path(configured)
    if p.is_dir() and (any(p.glob(must_contain)) if "*" in must_contain
                       else (p / must_contain).exists()):
        return p
    for base in (Path("/kaggle/input"), Path("data")):
        if not base.is_dir():
            continue
        for root, dirs, _ in os.walk(base):
            # Never descend into the competition mount: it holds ~700k DICOMs and none of
            # them is what we are looking for. Walking it turns a scan into a coffee break.
            dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
            r = Path(root)
            if any(r.glob(must_contain)):
                return r
    raise FileNotFoundError(f"no directory containing {must_contain!r} (tried {configured})")


def find_cache_files(key):
    """Indexes the cached studies, preferring cache directories whose path contains key.

    Walks the whole input tree (kernel outputs mount at varying depths) but never descends
    into the competition DICOM folders. Falls back to any directory of *.npz if key matches
    none, and to ./data/cache* locally.

    Args:
        key: Substring identifying the wanted cache version, e.g. "rsna-knee-cache-v3".

    Returns:
        A dict mapping StudyInstanceUID to the path of its .npz cache file.

    Raises:
        FileNotFoundError: No *.npz file exists under the search roots.
    """
    bases = [Path("/kaggle/input"), Path("data")]
    hits, all_hits = {}, {}
    for base in bases:
        if not base.is_dir():
            continue
        for root, dirs, fnames in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
            r = Path(root)
            npz = {f[:-4]: r / f for f in fnames if f.endswith(".npz")}
            if not npz:
                continue
            dirs[:] = []                # a cache dir is flat; nothing useful below it
            target = hits if key in str(r) else all_hits
            for k, v in npz.items():
                target.setdefault(k, v)
            print(f"cache dir: {r} ({len(npz)} files){'' if key in str(r) else '  [key not in path]'}")
    files = hits or all_hits
    if not files:
        raise FileNotFoundError(f"no cache/*.npz found under {bases} (key {key!r})")
    return files


cached = find_cache_files(CFG["CACHE_KEY"])
with np.load(next(iter(cached.values()))) as probe:
    cache_shape = probe["vol"].shape
CFG["N_SLOTS"] = int(cache_shape[0])
CFG["CACHE_SLICES"] = int(cache_shape[1])
CFG["IMG"] = int(cache_shape[2])
print(f"cache: {len(cached)} studies | slots {CFG['N_SLOTS']} x slices {CFG['CACHE_SLICES']} x {CFG['IMG']}px")
if CFG["SLOTS"] is not None:
    SLOT_SEL = np.asarray(CFG["SLOTS"], dtype=int)
    if SLOT_SEL.min() < 0 or SLOT_SEL.max() >= CFG["N_SLOTS"]:
        raise ValueError(f"SLOTS {CFG['SLOTS']} out of range for {CFG['N_SLOTS']} cache slots")
    CFG["N_SLOTS"] = len(SLOT_SEL)
    print(f"slot subset {SLOT_SEL.tolist()} -> {CFG['N_SLOTS']} slots")
else:
    SLOT_SEL = None

# Slice-direction fix (EXP-016): flip flags per study, applied BEFORE the slot subset so the
# table always indexes true cache slots. Missing rows mean "leave alone", which is also what a
# study whose correlation margin was ambiguous gets.
if CFG["SLICE_DIR_FIX"]:
    flip_table = pd.read_csv(CFG["SLICE_DIR_FIX"]).set_index("StudyInstanceUID")
    FLIP_SLOTS = sorted(int(c[4:]) for c in flip_table.columns if c.startswith("flip"))
    FLIP = {u: [s for s in FLIP_SLOTS if r[f"flip{s}"]] for u, r in flip_table.iterrows()}
    FLIP = {u: v for u, v in FLIP.items() if v}
    print(f"slice-direction fix: {len(FLIP)} of {len(flip_table)} studies flip at least one slot")
else:
    FLIP = None

CSV_DIR = find_dir(CFG["CSV_DIR"], CFG["FOLDS_CSV"])
OUT_DIR = Path(CFG["OUT_DIR"])
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("csvs :", CSV_DIR)

folds = pd.read_csv(CSV_DIR / CFG["FOLDS_CSV"])
weak = pd.read_csv(next(CSV_DIR.rglob(CFG["WEAK_CSV"])))

df = weak.merge(folds, on="StudyInstanceUID", how="inner")
df["has_cache"] = df.StudyInstanceUID.isin(cached)
print(f"weak labels {len(weak)} | with fold {len(df)} | cached {int(df.has_cache.sum())}")
df = df[df.has_cache].reset_index(drop=True)
df["is_gold"] = (df["source"] == "gold").astype(int)
print(df.groupby(["fold", "source"]).size().unstack(fill_value=0).to_string())

In [ ]:
FOLD = CFG["FOLD"]
train_df = df[df.fold != FOLD].reset_index(drop=True)
valid_df = df[df.fold == FOLD].reset_index(drop=True)

if CFG["DEBUG"]:
    n = CFG["DEBUG_STUDIES"]

    # Keep the gold rows in the debug subsets or the gold AUC block has nothing to score.
    def head_with_gold(d, k):
        """Shrinks a split to about k studies while keeping every gold row.

        Args:
            d: The split to shrink, with an is_gold column.
            k: Target number of rows; the gold rows are kept even if they exceed it.

        Returns:
            The shuffled subset, with a fresh index.
        """
        gold = d[d.is_gold == 1]
        rest = d[d.is_gold == 0].head(max(k - len(gold), 0))
        return pd.concat([gold, rest]).sample(frac=1, random_state=CFG["SEED"]).reset_index(drop=True)

    train_df = head_with_gold(train_df, int(n * 0.8))
    valid_df = head_with_gold(valid_df, int(n * 0.2))

print(f"fold {FOLD}: train {len(train_df)} ({train_df.is_gold.sum()} gold) | "
      f"valid {len(valid_df)} ({valid_df.is_gold.sum()} gold)")

## Dataset

No horizontal flips. The metric contains `Medial OA` / `Lateral OA` and medial / lateral
meniscus: mirroring a knee swaps those compartments and turns four of the twelve targets into
noise. Augmentation is limited to a slice-offset jitter and a mild intensity scale/shift.

In [ ]:
IS_VIT = "vit" in CFG["BACKBONE"]
IMG_IN = CFG["VIT_IMG"] if IS_VIT else (CFG["MODEL_IMG"] or CFG["IMG"])
print("model input side:", IMG_IN, "| cache side:", CFG["IMG"])
SLICE_IDX = np.arange(0, CFG["CACHE_SLICES"], CFG["SLICE_STRIDE"])
N_SLICE = len(SLICE_IDX)
print(f"{N_SLICE} slices per slot (stride {CFG['SLICE_STRIDE']} over {CFG['CACHE_SLICES']})")


# cv2 does both the affine augmentation and the ViT resize. Silently skipping them when it is
# missing would train a different model from the one the config describes, so fail at import.
try:
    import cv2
except ImportError as exc:
    raise ImportError("cv2 (opencv-python) is required for augmentation and resizing") from exc
cv2.setNumThreads(0)


def random_affine(x, rng):
    """Applies one random rotation, scale and shift to every image of a study.

    The same transform is used for all slots and slices, so the study stays internally
    aligned. Angles, scale and shift ranges come from AUG_ROT_DEG, AUG_SCALE and AUG_SHIFT.

    Args:
        x: Study volume of shape (slots, S, H, W), float in [0, 1].
        rng: The numpy Generator to draw the transform from.

    Returns:
        The warped volume, same shape and dtype as x.
    """
    h, w = x.shape[-2:]
    ang = rng.uniform(-CFG["AUG_ROT_DEG"], CFG["AUG_ROT_DEG"])
    sc = 1.0 + rng.uniform(-CFG["AUG_SCALE"], CFG["AUG_SCALE"])
    tx, ty = rng.uniform(-CFG["AUG_SHIFT"], CFG["AUG_SHIFT"], size=2) * np.array([w, h])
    mat = cv2.getRotationMatrix2D((w / 2, h / 2), ang, sc)
    mat[:, 2] += (tx, ty)
    flat = x.reshape(-1, h, w)
    out = np.empty_like(flat)
    for i in range(flat.shape[0]):
        out[i] = cv2.warpAffine(flat[i], mat, (w, h), flags=cv2.INTER_LINEAR,
                                borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    return out.reshape(x.shape)


class KneeCache(Dataset):
    """One cached study per item: the 2.5D image stack, its labels and its weights.

    Args:
        frame: Rows of the merged weak-label/fold table to serve.
        train: True to augment and to jitter the slice grid.

    Attributes:
        df: The served rows, re-indexed from 0.
        train: Whether augmentation is on.
        y: Soft targets of shape (n, 12).
        w: Per-study sample weight, GOLD_WEIGHT on gold rows.
        wl: Per-label confidence weight of shape (n, 12).
    """

    def __init__(self, frame, train):
        self.df = frame.reset_index(drop=True)
        self.train = train
        self.y = frame[TARGETS].to_numpy(np.float32)
        gold = frame["is_gold"].to_numpy() == 1
        self.w = np.where(gold, CFG["GOLD_WEIGHT"], 1.0).astype(np.float32)
        # Per-label weight: an LLM score near 0.5 is a hedge or a silence-prior and should
        # pull less than an asserted finding/negation. Gold rows are hard labels: weight 1.
        conf = 2.0 * np.abs(self.y - 0.5)
        wl = CFG["CONF_FLOOR"] + (1.0 - CFG["CONF_FLOOR"]) * conf if CFG["CONF_WEIGHT"] else np.ones_like(self.y)
        wl[gold] = 1.0
        self.wl = wl.astype(np.float32)

    def __len__(self):
        """Returns the number of studies served."""
        return len(self.df)

    def __getitem__(self, i):
        """Loads, augments and standardises one study.

        Every random draw comes from one freshly seeded Generator created here rather than
        from the global numpy RNG: DataLoader seeds torch per worker but not numpy, so the
        global RNG hands every worker the same jitter and the same intensity scale.

        Args:
            i: Row index into the served frame.

        Returns:
            A dict with "x" (slots, S, 3, H, W) float32, "present" (slots,), "y" (12,),
            "w" (scalar) and "wl" (12,), all torch tensors.
        """
        rng = np.random.default_rng()
        study = self.df.StudyInstanceUID.iloc[i]
        with np.load(cached[study]) as d:
            vol = d["vol"]                     # (slots, CACHE_SLICES, H, W) uint8
            present = d["present"].astype(np.float32)
            if FLIP is not None and study in FLIP:
                vol = vol.copy()
                for slot in FLIP[study]:
                    vol[slot] = vol[slot][::-1]
            if SLOT_SEL is not None:
                vol, present = vol[SLOT_SEL], present[SLOT_SEL]

        n = vol.shape[1]
        idx = SLICE_IDX
        if self.train:
            # Jitter which slices are seen, within the stride, so the model does not
            # memorise one fixed sampling grid.
            off = rng.integers(0, CFG["SLICE_STRIDE"])
            idx = np.clip(idx + off, 0, n - 1)

        if self.train and CFG["AUG"]:
            vol = random_affine(vol.astype(np.float32) / 255.0, rng)
            vol = np.clip(vol, 0, 1) ** np.float32(np.exp(rng.uniform(-CFG["AUG_GAMMA"], CFG["AUG_GAMMA"])))
            vol = (vol * 255.0).astype(np.float32)

        # 2.5D: channels are the true cache neighbours (i-1, i, i+1), clamped at the ends.
        # These are indexed out of the full stack BEFORE subsampling. Building the triple
        # after striding would give (i-2, i, i+2) at SLICE_STRIDE=2, make each side channel
        # a duplicate of the neighbouring sample's centre channel, and leave the odd cache
        # slices unread whenever the jitter offset is 0.
        x = np.stack([vol[:, np.clip(idx - 1, 0, n - 1)],
                      vol[:, idx],
                      vol[:, np.clip(idx + 1, 0, n - 1)]],
                     axis=2).astype(np.float32) / 255.0   # (slots, S, 3, H, W)

        if self.train:
            x = x * np.float32(rng.uniform(0.9, 1.1)) + np.float32(rng.uniform(-0.05, 0.05))
            x = np.clip(x, 0.0, 1.0)

        if IMG_IN != vol.shape[-1]:
            k, s_, c, h, w = x.shape
            flat = x.reshape(-1, h, w)
            x = np.stack([cv2.resize(f, (IMG_IN, IMG_IN), interpolation=cv2.INTER_AREA)
                          for f in flat]).reshape(k, s_, c, IMG_IN, IMG_IN)

        # ImageNet-ish standardisation on a greyscale image: one scalar mean/std is enough.
        x = (x - 0.449) / 0.226

        return {
            "x": torch.from_numpy(np.ascontiguousarray(x)),
            "present": torch.from_numpy(present),
            "y": torch.from_numpy(self.y[i]),
            "w": torch.tensor(self.w[i]),
            "wl": torch.from_numpy(self.wl[i]),
        }


train_ds = KneeCache(train_df, train=True)
valid_ds = KneeCache(valid_df, train=False)

train_dl = DataLoader(train_ds, batch_size=CFG["BATCH_SIZE"], shuffle=True, drop_last=True,
                      num_workers=CFG["NUM_WORKERS"], pin_memory=True, persistent_workers=CFG["NUM_WORKERS"] > 0)
valid_dl = DataLoader(valid_ds, batch_size=CFG["BATCH_SIZE"], shuffle=False,
                      num_workers=CFG["NUM_WORKERS"], pin_memory=True, persistent_workers=CFG["NUM_WORKERS"] > 0)

sample_batch = next(iter(train_dl))
print({k: tuple(v.shape) for k, v in sample_batch.items()})

## Model

In [ ]:
class AttnPool(nn.Module):
    """Gated attention over the slices of one slot (Ilse et al., attention MIL).

    One gate serves every slot and every label, so this pooling is label-agnostic and
    order-blind. See `LabelAttnHead` for the per-label alternative.

    Args:
        dim: Slice embedding width.
        hidden: Width of the gate MLP.
    """

    def __init__(self, dim, hidden):
        super().__init__()
        self.v = nn.Linear(dim, hidden)
        self.u = nn.Linear(dim, hidden)
        self.w = nn.Linear(hidden, 1)

    def forward(self, x):
        """Pools (B, S, D) slice embeddings into (B, D).

        Args:
            x: Slice embeddings of one slot per row.

        Returns:
            The pooled embedding and the (B, S) attention weights.
        """
        a = self.w(torch.tanh(self.v(x)) * torch.sigmoid(self.u(x)))   # (B, S, 1)
        a = a.softmax(dim=1)
        return (a * x).sum(dim=1), a.squeeze(-1)


class LabelAttnHead(nn.Module):
    """Per-label attention over every slice embedding in the study (attention-MIL).

    Each label owns one softmax over the K*S slice embeddings, so a finding visible on two
    slices of one slot is kept rather than averaged down by a gate shared across labels.
    Slices of absent slots are masked out of every softmax.

    Args:
        dim: Slice embedding width.
        hidden: Width of the attention MLP.
        n_slots: Number of cache slots K.
        n_out: Number of labels.
        slot_emb: Add a learned per-slot vector to each slice embedding before attention.
        dropout: Dropout inside the attention MLP.
    """

    def __init__(self, dim, hidden, n_slots, n_out, slot_emb, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.slot_emb = nn.Parameter(torch.zeros(n_slots, dim)) if slot_emb else None
        self.att = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Dropout(dropout),
                                 nn.Linear(hidden, n_out))
        self.cls_w = nn.Parameter(torch.empty(n_out, dim))
        self.cls_b = nn.Parameter(torch.zeros(n_out))
        nn.init.trunc_normal_(self.cls_w, std=0.02)

    def forward(self, f, present):
        """Pools (B, K, S, D) slice embeddings into (B, n_out) logits.

        Args:
            f: Slice embeddings, one per (slot, slice).
            present: (B, K) mask, 1 where the slot exists in the study.

        Returns:
            Logits of shape (B, n_out).
        """
        b, k, s, d = f.shape
        if self.slot_emb is not None:
            f = f + self.slot_emb[None, :, None, :]
        h = self.norm(f).reshape(b, k * s, d)
        a = self.att(h)                                                # (B, K*S, n_out)
        keep = (present > 0.5)[:, :, None].expand(b, k, s).reshape(b, k * s, 1)
        a = a.masked_fill(~keep, -1e4).softmax(dim=1)
        pooled = torch.einsum("bnl,bnd->bld", a, h)                    # (B, n_out, D)
        return (pooled * self.cls_w).sum(-1) + self.cls_b


class Model2p5D(nn.Module):
    """2.5D study classifier: one encoder per slice, pooled per slot, one logit per label.

    Args:
        cfg: The training CFG dict; reads BACKBONE, PRETRAINED, VIT_*, GRAD_CHECKPOINT,
            N_SLOTS, HEAD, HEAD_HIDDEN, SLOT_EMB, ATTN_DIM, DROPOUT, and for POS_EMB also
            CACHE_SLICES and SLICE_STRIDE.
    """

    def __init__(self, cfg):
        super().__init__()
        self.n_slots = cfg["N_SLOTS"]
        self.is_vit = "vit" in cfg["BACKBONE"]
        kw = {}
        if self.is_vit:
            kw["img_size"] = cfg["VIT_IMG"]
        elif "coatnet" in cfg["BACKBONE"]:
            # CoAtNet fixes its attention window at build time from img_size, so it must be
            # told the side it will be fed: MODEL_IMG when the dataset upsamples, else the
            # cache side.
            kw["img_size"] = cfg.get("MODEL_IMG") or cfg["IMG"]
        self.backbone = timm.create_model(
            cfg["BACKBONE"], pretrained=cfg["PRETRAINED"], num_classes=0, in_chans=3, **kw)
        if cfg["GRAD_CHECKPOINT"]:
            self.backbone.set_grad_checkpointing(True)
        dim = self.backbone.num_features
        if self.is_vit:
            # DINOv2 recipe: freeze everything but the last VIT_UNFREEZE blocks (+ final
            # norm); feature = [CLS ; mean of patch tokens]. VIT_UNFREEZE >= depth means a
            # genuinely full fine-tune, with patch_embed, cls_token and pos_embed trainable
            # too, so that a ViT can be compared fairly against a fully trained convnext.
            n_blocks = len(self.backbone.blocks)
            if cfg["VIT_UNFREEZE"] < n_blocks:
                for p in self.backbone.parameters():
                    p.requires_grad = False
                for blk in self.backbone.blocks[-cfg["VIT_UNFREEZE"]:]:
                    for p in blk.parameters():
                        p.requires_grad = True
                for p in self.backbone.norm.parameters():
                    p.requires_grad = True
            else:
                print(f"ViT: FULL fine-tune ({n_blocks} blocks, nothing frozen)")
            dim = dim * 2
        # A learned vector per (slot, sampled slice position), added before pooling. The pools
        # are order-blind and the right-knee mirror reverses sagittal slice order, so without
        # this the model cannot tell where in the stack a slice sits.
        self.pos_emb = None
        if cfg.get("POS_EMB"):
            n_pos = len(range(0, cfg["CACHE_SLICES"], cfg["SLICE_STRIDE"]))
            self.pos_emb = nn.Parameter(torch.zeros(self.n_slots, n_pos, dim))
        self.head_kind = cfg["HEAD"]
        if self.head_kind == "label_attn":
            self.head = LabelAttnHead(dim, cfg["HEAD_HIDDEN"], self.n_slots, len(TARGETS),
                                      cfg["SLOT_EMB"], cfg["DROPOUT"])
        else:
            self.pool = AttnPool(dim, cfg["ATTN_DIM"])
            self.drop = nn.Dropout(cfg["DROPOUT"])
            self.head = nn.Linear(dim * self.n_slots, len(TARGETS))

    def forward(self, x, present=None):
        """Scores a batch of studies.

        Args:
            x: uint8 or float images of shape (B, slots, S, 3, H, W).
            present: (B, slots) mask, 1 where the slot exists in the study.

        Returns:
            Logits of shape (B, 12).
        """
        b, k, s = x.shape[:3]
        flat = x.reshape(b * k * s, *x.shape[3:])
        if self.is_vit:
            t = self.backbone.forward_features(flat)                   # (N, 1+P, D)
            npre = self.backbone.num_prefix_tokens
            f = torch.cat([t[:, 0], t[:, npre:].mean(dim=1)], dim=1)   # (N, 2D)
        else:
            f = self.backbone(flat)                                    # (B*k*S, D)
        f = f.reshape(b, k, s, -1)
        if self.pos_emb is not None:
            f = f + self.pos_emb[None, :, :s]
        if self.head_kind == "label_attn":
            if present is None:
                present = torch.ones(b, k, device=f.device)
            return self.head(f, present)
        f = f.reshape(b * k, s, -1)
        emb, _ = self.pool(f)                                          # (B*k, D)
        emb = emb.reshape(b, k, -1)
        if present is not None:
            # A missing slot is all zeros in the cache; zero its embedding too, so the head
            # sees "absent" rather than "the embedding of a black volume".
            emb = emb * present.unsqueeze(-1)
        return self.head(self.drop(emb.reshape(b, -1)))


## Loss, optimizer, schedule

In [ ]:
def weighted_bce(logits, target, sample_w, label_w=None):
    """BCE on soft targets, weighted per study (gold rows count GOLD_WEIGHT x) and per label.

    The denominator is the batch size, not sample_w.sum(). Dividing by the weight sum
    renormalises within the batch and cancels most of the upweight -- an all-gold batch
    would come out identical to an all-weak one, leaving GOLD_WEIGHT=2.0 realised as
    roughly 1.0-1.33x. A fixed denominator makes a gold study contribute exactly twice
    the gradient of a weak one, which is what the config says.

    Args:
        logits: Raw model outputs of shape (B, 12).
        target: Soft targets in [0, 1] of shape (B, 12).
        sample_w: Per-study weight of shape (B,).
        label_w: Per-label weight of shape (B, 12), or None for no label weighting.

    Returns:
        The scalar loss for the batch.
    """
    loss = F.binary_cross_entropy_with_logits(logits, target, reduction="none")  # (B, 12)
    if label_w is not None:
        loss = loss * label_w
    return (loss.mean(dim=1) * sample_w).sum() / logits.shape[0]


head_params = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith(("head", "pool", "pos_emb"))]
if model.is_vit:
    # Layer-wise LR decay over the unfrozen blocks: the last block gets the full backbone LR,
    # each earlier one VIT_LLRD x less. Final norm rides with the last block.
    n_blk = len(model.backbone.blocks)
    groups = [{"params": head_params, "lr": CFG["VIT_LR_HEAD"]}]
    for i, blk in enumerate(model.backbone.blocks):
        ps = [p for p in blk.parameters() if p.requires_grad]
        if ps:
            groups.append({"params": ps, "lr": CFG["VIT_LR_BACKBONE"] * CFG["VIT_LLRD"] ** (n_blk - 1 - i)})
    groups.append({"params": [p for p in model.backbone.norm.parameters() if p.requires_grad], "lr": CFG["VIT_LR_BACKBONE"]})
    optimizer = torch.optim.AdamW(groups, weight_decay=CFG["WEIGHT_DECAY"])
    print("ViT param groups:", [(len(g["params"]), f"{g['lr']:.2e}") for g in groups])
else:
    trunk_params = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith(("head", "pool", "pos_emb"))]
    optimizer = torch.optim.AdamW(
        [{"params": trunk_params, "lr": CFG["LR"]},
         {"params": head_params, "lr": CFG["LR"] * CFG["HEAD_LR_MULT"]}],
        weight_decay=CFG["WEIGHT_DECAY"])

steps_per_epoch = max(len(train_dl) // CFG["GRAD_ACCUM"], 1)
stage1_steps = steps_per_epoch * CFG["STAGE1_EPOCHS"]
# The cosine runs over the epochs AFTER the frozen stage, and only over those (B1). It used
# to be defined over that count and then grown by stage1_steps, while lr_head/lr_trunk kept
# calling it with (step - stage1_steps): the argument never reached the end of the range, so
# an 8-epoch run with 2 frozen epochs finished at ~15% of peak LR instead of 0. Keeping the
# two counts apart fixes that and leaves STAGE1_EPOCHS=0 bit-identical, since cosine_steps
# is then exactly the old total_steps.
cosine_steps = steps_per_epoch * (CFG["EPOCHS"] - CFG["STAGE1_EPOCHS"])
total_steps = stage1_steps + cosine_steps
warmup = max(int(cosine_steps * CFG["WARMUP_FRAC"]), 1)


def lr_at(step):
    """The LR multiplier of the warmup-then-cosine schedule.

    Args:
        step: Step index within the cosine stage, 0-based.

    Returns:
        A multiplier in [0, 1]: linear warmup, then a cosine reaching 0 at cosine_steps.
    """
    if step < warmup:
        return (step + 1) / warmup
    prog = (step - warmup) / max(cosine_steps - warmup, 1)
    return 0.5 * (1 + np.cos(np.pi * min(prog, 1.0)))


def lr_head(step):
    """The head LR multiplier: the schedule runs from step 0, through both stages.

    Args:
        step: Global optimizer step, 0-based.

    Returns:
        The multiplier applied to the head param group's base LR.
    """
    return lr_at(step) if step < stage1_steps else lr_at(step - stage1_steps)


def lr_trunk(step):
    """The backbone LR multiplier: zero while the trunk is frozen, then the schedule.

    Args:
        step: Global optimizer step, 0-based.

    Returns:
        The multiplier applied to the trunk param groups' base LRs.
    """
    return 0.0 if step < stage1_steps else lr_at(step - stage1_steps)


# Param-group order: ViT = [head, blocks..., norm]; CNN = [trunk, head].
is_head_group = [g["params"] is head_params
                 or (len(g["params"]) == len(head_params)
                     and g["params"][0] is head_params[0])
                 for g in optimizer.param_groups]
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, [lr_head if h else lr_trunk for h in is_head_group])
if stage1_steps:
    print(f"two-stage: {CFG['STAGE1_EPOCHS']} head-only epoch(s) = {stage1_steps} steps, then full fine-tune")
scaler = amp_scaler(USE_AMP)
print(f"{steps_per_epoch} optimizer steps/epoch x {CFG['EPOCHS']} epochs = {total_steps} "
      f"(warmup {warmup})")
print(f"LR multiplier at the last step ({total_steps - 1}): "
      f"head {lr_head(total_steps - 1):.4f} | trunk {lr_trunk(total_steps - 1):.4f}")

## Metric

In [ ]:
def per_label_auc(y_true_bin, y_pred):
    """Scores each of the 12 labels with ROC AUC.

    Args:
        y_true_bin: Binary labels of shape (n, 12).
        y_pred: Predicted scores of shape (n, 12).

    Returns:
        An array of 12 AUCs, NaN where only one class is present for that label.
    """
    out = []
    for j in range(len(TARGETS)):
        yt = y_true_bin[:, j]
        if len(np.unique(yt)) < 2:
            out.append(np.nan)
        else:
            out.append(roc_auc_score(yt, y_pred[:, j]))
    return np.array(out, dtype=float)


def safe_nanmean(a):
    """Averages the non-NaN entries.

    Args:
        a: Any array-like of floats.

    Returns:
        The mean of the finite entries, or NaN when every entry is NaN. Unlike
        np.nanmean this raises no RuntimeWarning on an all-NaN input.
    """
    a = np.asarray(a, dtype=float)
    ok = ~np.isnan(a)
    return float(a[ok].mean()) if ok.any() else float("nan")


def report_auc(name, aucs):
    """Prints the per-label AUCs and their mean.

    Args:
        name: Label for the printed block, e.g. "fold 0 vs GOLD only".
        aucs: The 12 per-label AUCs, NaN where unscorable.

    Returns:
        The mean over the scorable labels, NaN if there are none.
    """
    ok = ~np.isnan(aucs)
    mean_auc = safe_nanmean(aucs)
    print(f"\n{name}: mean AUC = {mean_auc:.4f} over {int(ok.sum())}/12 labels")
    print("  " + "  ".join(f"{t}={a:.3f}" if not np.isnan(a) else f"{t}=--"
                           for t, a in zip(TARGETS, aucs)))
    return mean_auc

## Train / validate

In [ ]:
@torch.no_grad()
def evaluate(loader, frame):
    """Runs the model over a validation split and scores it against weak and gold labels.

    Args:
        loader: DataLoader over the split, in frame order and unshuffled.
        frame: The split's rows, carrying the soft targets and the is_gold flag.

    Returns:
        A tuple (preds, mean_weak, weak_auc, gold_auc): the (n, 12) probabilities, the mean
        AUC against binarised weak labels, the 12 weak AUCs and the 12 gold AUCs (all NaN
        when the split holds fewer than 4 gold rows).
    """
    model.eval()
    preds = []
    for batch in tqdm(loader, desc="valid", leave=False):
        x = batch["x"].to(DEVICE, non_blocking=True)
        pres = batch["present"].to(DEVICE, non_blocking=True)
        with amp_autocast(USE_AMP):
            logits = model(x, pres)
        preds.append(torch.sigmoid(logits.float()).cpu().numpy())
    preds = np.concatenate(preds, axis=0)

    soft = frame[TARGETS].to_numpy(np.float32)
    weak_auc = per_label_auc((soft > 0.5).astype(int), preds)
    mean_weak = report_auc(f"fold {FOLD} vs weak labels (>0.5), n={len(frame)}", weak_auc)

    gold_mask = frame["is_gold"].to_numpy() == 1
    gold_auc = np.full(len(TARGETS), np.nan)
    if gold_mask.sum() >= 4:
        gold_auc = per_label_auc((soft[gold_mask] > 0.5).astype(int), preds[gold_mask])
        report_auc(f"fold {FOLD} vs GOLD only, n={int(gold_mask.sum())}", gold_auc)
    else:
        print(f"\ngold rows in fold {FOLD}: {int(gold_mask.sum())} -- too few to score")

    return preds, mean_weak, weak_auc, gold_auc

In [ ]:
import copy
budget = CFG["TIME_BUDGET_H"] * 3600
# best_preds must exist even if every epoch scores NaN -- which a tiny DEBUG validation
# split can genuinely do, when no label has both classes present. Without this the run
# finishes, writes no checkpoint, and cell "OOF predictions" dies on a NameError.
best = {"score": -np.inf, "epoch": -1}
best_preds = None
history = []
stop = False
swa_state, n_avg, swa_preds = None, 0, None


def swa_update(state, n_avg, model):
    """Folds the model's current weights into a running equal-weight average.

    Float tensors (parameters, BatchNorm running stats) are averaged; integer buffers such as
    num_batches_tracked are copied from the latest weights. Done by hand because torch 2.6
    rejects AveragedModel(use_buffers=True) on the integer buffers.
    """
    current = {k: v.detach().clone() for k, v in model.state_dict().items()}
    if state is None:
        return current, 1
    n = n_avg + 1
    for k, v in current.items():
        if v.is_floating_point():
            state[k].mul_((n - 1) / n).add_(v, alpha=1.0 / n)
        else:
            state[k].copy_(v)
    return state, n


for epoch in range(CFG["EPOCHS"]):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running, seen = 0.0, 0
    bar = tqdm(train_dl, desc=f"epoch {epoch}", smoothing=0.05)

    for i, batch in enumerate(bar):
        x = batch["x"].to(DEVICE, non_blocking=True)
        pres = batch["present"].to(DEVICE, non_blocking=True)
        y = batch["y"].to(DEVICE, non_blocking=True)
        w = batch["w"].to(DEVICE, non_blocking=True)
        wl = batch["wl"].to(DEVICE, non_blocking=True)

        with amp_autocast(USE_AMP):
            loss = weighted_bce(model(x, pres), y, w, wl)

        scaler.scale(loss / CFG["GRAD_ACCUM"]).backward()

        if (i + 1) % CFG["GRAD_ACCUM"] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["GRAD_CLIP"])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        running += float(loss.detach()) * len(y)
        seen += len(y)
        bar.set_postfix(loss=f"{running / seen:.4f}",
                        lr=f"{scheduler.get_last_lr()[0]:.2e}",
                        h=f"{(time.time() - T0) / 3600:.2f}")

        if time.time() - T0 > budget:
            print(f"\ntime budget hit mid-epoch {epoch}; validating and stopping.")
            stop = True
            break

    preds, mean_weak, weak_auc, gold_auc = evaluate(valid_dl, valid_df)
    history.append({"epoch": epoch, "train_loss": running / max(seen, 1),
                    "mean_weak_auc": mean_weak, "mean_gold_auc": safe_nanmean(gold_auc)})
    print(f"epoch {epoch}: loss {running / max(seen, 1):.4f} | mean weak AUC {mean_weak:.4f}")

    # NaN never counts as an improvement, but the first epoch is saved regardless so a
    # run whose validation cannot be scored still leaves usable weights behind.
    improved = (not np.isnan(mean_weak)) and mean_weak > best["score"]
    if improved or best_preds is None:
        best = {"score": mean_weak if improved else best["score"], "epoch": epoch}
        torch.save({"model": model.state_dict(), "cfg": CFG, "fold": FOLD,
                    "epoch": epoch, "mean_weak_auc": mean_weak,
                    "weak_auc": weak_auc.tolist(), "gold_auc": gold_auc.tolist()},
                   OUT_DIR / f"best_fold{FOLD}.pt")
        best_preds = preds
        print(f"  saved best_fold{FOLD}.pt" + ("" if improved else " (fallback: score is NaN)"))

    # Equal-weight average of the epoch-end weights over the last SWA_EPOCHS epochs.
    # use_buffers averages the BatchNorm statistics too, so the averaged model needs no
    # extra pass over the training set to recalibrate them.
    if CFG["SWA_EPOCHS"] and epoch >= CFG["EPOCHS"] - CFG["SWA_EPOCHS"]:
        swa_state, n_avg = swa_update(swa_state, n_avg, model)

    gc.collect()
    torch.cuda.empty_cache()
    if stop:
        break

if swa_state is not None:
    print(f"\nSWA: average of the last {n_avg} epoch-end weights")
    eval_model = model
    model = copy.deepcopy(model)  # evaluate() scores the global `model`
    model.load_state_dict(swa_state)
    swa_preds, mean_weak, weak_auc, gold_auc = evaluate(valid_dl, valid_df)
    model = eval_model
    history.append({"epoch": f"swa{n_avg}", "train_loss": float("nan"),
                    "mean_weak_auc": mean_weak, "mean_gold_auc": safe_nanmean(gold_auc)})
    print(f"swa{n_avg}: mean weak AUC {mean_weak:.4f}")
    torch.save({"model": swa_state, "cfg": CFG, "fold": FOLD,
                "epoch": f"swa{n_avg}", "mean_weak_auc": mean_weak,
                "weak_auc": weak_auc.tolist(), "gold_auc": gold_auc.tolist()},
               OUT_DIR / f"swa_fold{FOLD}.pt")

print(f"\nbest: epoch {best['epoch']} | mean weak AUC {best['score']:.4f} | "
      f"{(time.time() - T0) / 3600:.2f} h elapsed")
print("CAVEAT: model selection and the headline number above are against LLM weak labels, "
      "not gold. Treat them as a training-progress signal only -- they are NOT comparable "
      "to the leaderboard, and they inherit whatever the extractor got wrong.")

## OOF predictions and run log

In [ ]:
if best_preds is None:
    # Only reachable if the epoch loop never ran (EPOCHS=0, or the time budget was already
    # spent). Nothing to write, and no reason to raise on the way out.
    print("no validation predictions were produced -- nothing to write.")
    oof = pd.DataFrame(columns=["StudyInstanceUID"] + TARGETS)
else:
    if swa_preds is not None:
        oof_swa = pd.DataFrame(swa_preds, columns=TARGETS)
        oof_swa.insert(0, "StudyInstanceUID", valid_df.StudyInstanceUID.to_numpy())
        oof_swa["fold"] = FOLD
        oof_swa["source"] = valid_df["source"].to_numpy()
        oof_swa.to_csv(OUT_DIR / f"oof_swa_fold{FOLD}.csv", index=False)
    oof = pd.DataFrame(best_preds, columns=TARGETS)
    oof.insert(0, "StudyInstanceUID", valid_df.StudyInstanceUID.to_numpy())
    oof["fold"] = FOLD
    oof["source"] = valid_df["source"].to_numpy()
    oof.to_csv(OUT_DIR / f"oof_fold{FOLD}.csv", index=False)

if history:
    pd.DataFrame(history).to_csv(OUT_DIR / f"history_fold{FOLD}.csv", index=False)
    print(pd.DataFrame(history).to_string(index=False))
print("\nwrote:", [p.name for p in sorted(OUT_DIR.glob(f"*fold{FOLD}*"))])
oof.head()

## Next steps

- Rerun with `FOLD = 1..4` to complete the OOF set, then pool all five folds *before*
  computing AUC — a 58-study gold set cannot arbitrate a single split (COMPETITION_SPEC §6).
- Log the run in `experiments/LEDGER.md` with wall-clock and mean AUC.
- Follow-ups with the highest EV against this baseline: IDEAS #22 (knee-region cropping —
  cheaper *and* likely better), #15 (backbone head-to-head on this exact fold), #16
  (cross-attention across slots instead of concatenation).